In [1]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, WhisperTokenizer,pipeline, WhisperProcessor, WhisperForConditionalGeneration
from datasets import load_dataset
import pandas as pd
import numpy as np

In [2]:
torch.cuda.empty_cache()

In [3]:
df_audio = pd.read_parquet('./data/parquets/testing_trained.parquet.gzip')
df_audio['snr_25_testing_trained']
df_audio['audio_SNR_25_path']
df_audio['SNR25_models_testing'] = df_audio['audio_SNR_25_path'].str.replace('\\', '/')

In [4]:
models= ["eryk7381/whisper-med-pol-car-15000", "eryk7381/whisper-med-pol-car-2500", "eryk7381/whisper-med-pol-car-10000", "eryk7381/whisper-med-pol-car-3500"]
columns = ["model15k", "model2_5k", "model10k", "model3_5k"]

columns_counter = 0


for model_id in models:

    temp_list = []

    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
  
    model = AutoModelForSpeechSeq2Seq.from_pretrained(model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True)
    model.to(device)

    processor = AutoProcessor.from_pretrained(model_id)

    pipe = pipeline(
        "automatic-speech-recognition",
        model=model,
        tokenizer=processor.tokenizer,
        feature_extractor=processor.feature_extractor,
        torch_dtype=torch_dtype,
        device=device,
        chunk_length_s=30
    )
    for audio in df_audio['snr_25_testing_trained'].to_list():
        result = pipe(audio)
        temp_list.append(result["text"])

    df_audio[columns[columns_counter]] = temp_list

    columns_counter = columns_counter + 1

    torch.cuda.empty_cache()

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
c:\Users\Eryk\anaconda3\envs\Magisterka\Lib\site-packages\transformers\models\whisper\modeling_whisper.py:697: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
c:\Users\Eryk\anaconda3\envs\Magisterka\Lib\site-packages\transformers\pipelines\base.py:1123: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
c:\Users\Eryk\anaconda3\envs\Magisterka\Lib\site-packages\transformers\pipelines\base.py:1123: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to m

In [ ]:
df_audio.to_parquet('./data/parquets/trained_results_newest.parquet.gzip', compression = 'gzip')

In [ ]:
import os
os.system('shutdown /s /t 0')